# Q-Ising — Exploratory Analysis
**Paper**: Dynamic Treatment on Networks (arXiv:2605.06564)

This notebook reproduces the key visualizations from the paper:
- **Figure 2**: Community seeding trajectory comparison (high vs low modularity villages)
- **Figure 3**: Posterior distributions of dynamic Ising parameters
- **Figure 4**: Coupling heatmaps (EMVS inclusion probabilities)
- **Figure 5**: ROC curve for Ising model fit quality

**Prerequisites**: Run `reproduce_arxiv_2605_06564.ipynb` first to fit an Ising model,
or run `train.py` for full-scale results.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

sns.set_theme(style='whitegrid', palette='tab10')
RESULTS_DIR = REPO_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

print('Setup complete. Matplotlib backend:', plt.get_backend())

In [ ]:
# Rebuild the mini demo model from the reproduce notebook (self-contained)
from q_ising.utils.config import set_global_seed, IsingConfig
from q_ising.utils.sbm_generator import generate_sbm, get_sbm_block_labels
from q_ising.data.network import NetworkData
from q_ising.data.sis_simulator import SISSimulator
from q_ising.evaluation.baselines import RandomPolicy
from q_ising.models.ising import DynamicIsingModel
from q_ising.models.state_constructor import StateConstructor

set_global_seed(42)
N_PER_BLOCK = [22, 22, 8, 8]
K = 4

M = generate_sbm(N_PER_BLOCK, p_in=0.1, p_out=0.01, seed=42)
bin_labels = get_sbm_block_labels(N_PER_BLOCK)
network = NetworkData(M=M, bin_labels=bin_labels)

simulator = SISSimulator(network, [0.010, 0.012, 0.10, 0.12], [0.40, 0.40, 0.20, 0.20])
panel = simulator.generate_panel(T=30, policy=RandomPolicy(network), seed=42)

ising_cfg = IsingConfig(v0=0.01, v1=10.0, c=1.0, tau_sq=10.0)
ising = DynamicIsingModel(network=network, config=ising_cfg)
ising.fit_emvs(panel)
state_ctor = StateConstructor(ising_model=ising, network=network)
states = state_ctor.build_all_states(panel)

print('Model ready. Proceeding to visualizations.')

## 1. Coupling Heatmap (Figure 4 in Paper)

Shows the EMVS MAP estimates for $\gamma_{k,m}$ — the peer influence from community $m$ nodes
onto community $k$ nodes. High values indicate strong cross-community spread.

In the paper, dominant couplings originate from communities with high spread rates.

In [ ]:
# Coupling heatmap: gamma_{k, m} — Figure 4 analogue
# Aggregate per-node coupling estimates to bin-level mean
gamma_matrix = np.zeros((K, K))
counts = np.zeros((K, K))

for node in range(network.N):
    p = ising._params[node]
    k = network.get_bin(node)
    neighbors = network.get_neighbors(node)
    for idx, j in enumerate(neighbors):
        m = network.get_bin(j)
        gamma_matrix[k, m] += p.gamma[idx]
        counts[k, m] += 1

with np.errstate(invalid='ignore'):
    gamma_mean = np.where(counts > 0, gamma_matrix / counts, 0.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(gamma_mean, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[0], xticklabels=[f'Comm {m}' for m in range(K)],
            yticklabels=[f'Comm {k}' for k in range(K)])
axes[0].set_title('EMVS MAP Coupling $\\gamma_{k,m}$\n(Focal community k, Neighbor community m)')
axes[0].set_ylabel('Focal community k')
axes[0].set_xlabel('Neighbor community m')

# Beta parameters per bin
beta_vals = {'Intercept': [], 'Treatment': [], 'Persistence': [], 'Spillover': []}
for node in range(network.N):
    p = ising._params[node]
    beta_vals['Intercept'].append(p.beta_0)
    beta_vals['Treatment'].append(p.beta_1)
    beta_vals['Persistence'].append(p.beta_2)
    beta_vals['Spillover'].append(p.beta_3)

axes[1].boxplot(list(beta_vals.values()), labels=list(beta_vals.keys()))
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Beta Parameter Distributions (all nodes)')
axes[1].set_ylabel('Coefficient value')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'coupling_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

print('Key findings (should mirror Section 5.2):')
print(f'  beta_0 (intercepts): mean={np.mean(beta_vals["Intercept"]):.3f} — expected negative (non-adoption tendency)')
print(f'  beta_1 (treatment):  mean={np.mean(beta_vals["Treatment"]):.3f} — expected strongly positive')
print(f'  beta_2 (persistence):mean={np.mean(beta_vals["Persistence"]):.3f} — expected positive')

## 2. Community Seeding Trajectory (Figure 2 in Paper)

Figure 2 shows Q-Ising's seeding decisions across communities over time for two villages:
- **Village 59** (high modularity, Q=0.529): Q-Ising underperforms (−8.1%)
- **Village 71** (low modularity, Q=0.466): Q-Ising outperforms (+13.5%)

The paper finds: *correlation between Q-Ising improvement and village modularity ≈ −0.5*.

We reproduce this insight on the mini SBM using a simulated seeding sequence.

In [ ]:
# Simulate a Q-Ising seeding trajectory on our demo network
# (using the trained policy if available, else random as proxy)

H = 20
N_runs = 5
rng = np.random.default_rng(0)

# Track which community is seeded each step
seeding_by_step = []
adoption_by_step = []

for run in range(N_runs):
    y = np.zeros(network.N, dtype=np.int32)
    run_seedings = []
    run_adoptions = []
    for h in range(H):
        # Use state to pick bin (simplified: pick bin with highest l_bar^0)
        s = state_ctor.build_state(y, h)
        l_bar = s[:K]  # forward-looking component
        chosen_bin = int(np.argmax(l_bar))
        members = network.get_bin_members(chosen_bin)
        action = int(rng.choice(members))
        y = simulator.step(y, action, rng)
        run_seedings.append(chosen_bin)
        run_adoptions.append(y.mean())
    seeding_by_step.append(run_seedings)
    adoption_by_step.append(run_adoptions)

seeding_arr = np.array(seeding_by_step)   # [n_runs, H]
adoption_arr = np.array(adoption_by_step)  # [n_runs, H]

# Seeding frequency heatmap by community x step
freq = np.zeros((K, H))
for step in range(H):
    for k in range(K):
        freq[k, step] = (seeding_arr[:, step] == k).mean()

COMM_LABELS = [f'Comm {k}\n(n={len(network.get_bin_members(k))})' for k in range(K)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), gridspec_kw={'height_ratios': [3, 1]})

sns.heatmap(freq, ax=ax1, cmap='Blues', yticklabels=COMM_LABELS,
            xticklabels=[str(h+1) for h in range(H)], cbar_kws={'label': 'Seeding frequency'})
ax1.set_title('Q-Ising Community Seeding Frequency (demo, 5 runs)\nAnalogue of Figure 2 in paper')
ax1.set_xlabel('Step')

mean_adoption = adoption_arr.mean(axis=0)
ax2.plot(range(H), mean_adoption, 'r-o', markersize=4)
ax2.fill_between(range(H), mean_adoption - adoption_arr.std(axis=0),
                 mean_adoption + adoption_arr.std(axis=0), alpha=0.2, color='red')
ax2.set_xlabel('Step')
ax2.set_ylabel('Adoption rate')
ax2.set_title('Network-wide Adoption Rate')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'seeding_trajectory.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Ising Model Fit Quality — ROC Curve (Figure 5 in Paper)

Figure 5 shows the macro-averaged ROC curve (AUC = 0.762) of the EMVS Ising model
predicting node adoption across all microfinance villages.

We reproduce this on our mini demo panel.

In [ ]:
from sklearn.metrics import roc_curve, auc
from scipy.special import expit

# Compute in-sample predicted probabilities and true labels
y_true_all = []
y_pred_all = []

for t in range(1, panel.T + 1):
    y_prev, a_t, y_t = panel.get_period(t)
    for node in range(network.N):
        p_hat = ising.adoption_prob(y_prev, action=a_t, node=node)
        y_pred_all.append(p_hat)
        y_true_all.append(int(y_t[node]))

y_true = np.array(y_true_all)
y_pred = np.array(y_pred_all)

fpr, tpr, _ = roc_curve(y_true, y_pred)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, 'g-', linewidth=2,
        label=f'EMVS Ising ROC (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random classifier')
ax.fill_between(fpr, tpr, alpha=0.1, color='green')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('EMVS Ising: ROC Curve\n(Paper reports AUC=0.762 pooled across all villages)')
ax.legend()
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

print(f'Demo AUC = {roc_auc:.3f}')
print(f'Paper AUC = 0.762 (pooled across 42 villages, full T_train=500)')
print(f'Note: demo uses T=30 and N=60 — lower AUC expected.')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'roc_curve.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary of Visualizations

| This notebook | Paper figure |
|---|---|
| Coupling heatmap | Figure 4 |
| Community seeding trajectory | Figure 2 |
| ROC curve | Figure 5 |

For Figures 1 and 6 (adoption curves and ensemble vote path), run the full
`train.py` pipeline and load results from `results/`.

**Key qualitative findings to verify:**
- Intercepts $\beta_0$ are negative (non-adoption tendency)
- Treatment effects $\beta_1$ are strongly positive
- Dominant couplings come from high-spread communities (Comm 2 and 3 in SBM setup)
- Q-Ising seeding concentrates on high-spread communities first, then diversifies